# hourly_patterns_table

In this notebook I will populate the hourly_patterns table. It has the following columns:
  - pattern_id serial [pk]
  - station_id varchar(50) [not null, ref: > stations.station_id]
  - day_of_week integer 
  - hour_of_day integer 
  - avg_trips_started decimal(6,2)
  - avg_trips_ended decimal(6,2)
  - avg_net_flow decimal(6,2)
  - avg_bikes_available decimal(5,2)

In [ ]:
import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import execute_values
import requests


# Connect to the citibike database on the Lightsail instance

In [ ]:
DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')


In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

### trips_started table: trips and stations tables are joined,  has the station, day of week and hour of day extracted from the started at date, and counts distict days for that station for that day

In [ ]:
cursor.execute(
    '''

    SELECT 
        s.station_id, 
        EXTRACT(DOW FROM t.started_at)::INTEGER AS dow, 
        EXTRACT(HOUR FROM t. started_at)::INTEGER AS hr, 
        COUNT(*) AS total_starts,
        COUNT(DISTINCT DATE(t.started_at)) AS days_sampled
    FROM trips t
    JOIN stations s ON s.short_name= t.start_station_id
    WHERE t.started_at IS NOT NULL
    GROUP BY s.station_id, dow, hr
    LIMIT 10;
    ''')
cursor.fetchall()


### Same as the trips started but for the trip_ends, this table uses the ended_at date from trips

In [ ]:
cursor.execute(
    '''

    SELECT  
            s.station_id, 
            EXTRACT(DOW FROM t.ended_at)::INTEGER AS dow, 
            EXTRACT(HOUR FROM t.ended_at)::INTEGER AS hr, 
            COUNT(*) AS total_ends,
            COUNT(DISTINCT DATE(t.ended_at)) AS days_sampled
    FROM  trips t
    JOIN stations s ON s.short_name= t.end_station_id
    WHERE t.ended_at IS NOT NULL
    GROUP BY s.station_id, dow, hr
    LIMIT 10;
    ''')
cursor.fetchall()

### this is the availability table where the station_id, day of week and hour and the avg bikes available at a station on that day and hour 

In [ ]:
cursor.execute(
    '''
    SELECT 
        station_id,
        EXTRACT(DOW FROM captured_at)::INTEGER AS dow,
        EXTRACT(HOUR FROM captured_at)::INTEGER AS hr,
        	ROUND(AVG(num_bikes_available), 2) AS avg_bikes_available
    FROM availability_snapshots
    WHERE is_installed = true AND is_renting = true
    GROUP BY station_id, dow, hr
    LIMIT 10;
''')
cursor.fetchall()

### all three tables are joined together using a union which will create a record of all the stations on 7 days a week and 24 hours a day with no duplicates

In [ ]:
cursor.execute('''
    WITH trip_starts AS(
        SELECT 
    s.station_id, 
        EXTRACT(DOW FROM t.started_at)::INTEGER AS dow, 
        EXTRACT(HOUR FROM t. started_at)::INTEGER AS hr, 
        COUNT(*) AS total_starts,
        COUNT(DISTINCT DATE(t.started_at))  AS days_sampled
    FROM trips t
    JOIN stations s ON s.short_name= t.start_station_id
    WHERE t.started_at IS NOT NULL
    GROUP BY s.station_id, dow, hr
    ),
    trip_ends AS (
        SELECT 
        s.station_id, 
        EXTRACT(DOW FROM t.ended_at)::INTEGER AS dow, 
        EXTRACT(HOUR FROM t.ended_at)::INTEGER AS hr, 
        COUNT(*) AS total_ends,
        COUNT(DISTINCT DATE(t.ended_at)) AS days_sampled
    
    FROM  trips t
    JOIN stations s ON s.short_name= t.end_station_id
    WHERE t.ended_at IS NOT NULL
    GROUP BY s.station_id, dow, hr
    ),
    availability_metrics AS(
        SELECT 
        station_id,
        EXTRACT(DOW FROM captured_at)::INTEGER AS dow,
        EXTRACT(HOUR FROM captured_at)::INTEGER AS hr,
        	ROUND(AVG(num_bikes_available), 2) AS avg_bikes_available
    FROM availability_snapshots
    WHERE is_installed = true AND is_renting = true
    GROUP BY station_id, dow, hr
    ),
    all_tables AS(
    SELECT station_id, dow,hr FROM trip_starts
    UNION
    SELECT station_id, dow,hr FROM trip_ends
    UNION
    SELECT station_id, dow,hr FROM availability_metrics
    )
    SELECT * 
    from all_tables
    limit 10;
    '''
)
cursor.fetchall()
    

### Extract all station_id, day of week and hour, and the avg_net_flow where avg(trip_ends) - avg(trip_starts ) is calculated

In [ ]:
cursor.execute('''
    WITH trip_starts AS(
        SELECT 
    s.station_id, 
        EXTRACT(DOW FROM t.started_at)::INTEGER AS dow, 
        EXTRACT(HOUR FROM t. started_at)::INTEGER AS hr, 
        COUNT(*) AS total_starts,
        COUNT(DISTINCT DATE(t.started_at))  AS days_sampled
    FROM trips t
    JOIN stations s ON s.short_name= t.start_station_id
    WHERE t.started_at IS NOT NULL
    GROUP BY s.station_id, dow, hr
    ),
    trip_ends AS (
        SELECT 
        s.station_id, 
        EXTRACT(DOW FROM t.ended_at)::INTEGER AS dow, 
        EXTRACT(HOUR FROM t.ended_at)::INTEGER AS hr, 
        COUNT(*) AS total_ends,
        COUNT(DISTINCT DATE(t.ended_at)) AS days_sampled
    
    FROM  trips t
    JOIN stations s ON s.short_name= t.end_station_id
    WHERE t.ended_at IS NOT NULL
    GROUP BY s.station_id, dow, hr
    ),
    availability_metrics AS(
        SELECT 
        station_id,
        EXTRACT(DOW FROM captured_at)::INTEGER AS dow,
        EXTRACT(HOUR FROM captured_at)::INTEGER AS hr,
        ROUND(AVG(num_bikes_available), 2) AS avg_bikes_available
    FROM availability_snapshots
    WHERE is_installed = true AND is_renting = true
    GROUP BY station_id, dow, hr
    ),
    all_tables AS(
    SELECT station_id, dow,hr FROM trip_starts
    UNION
    SELECT station_id, dow,hr FROM trip_ends
    UNION
    SELECT station_id, dow,hr FROM availability_metrics
        )
    
    SELECT 
        allt.station_id, 
        allt.dow AS day_of_week, 
        allt.hr AS hour_of_day,
        	COALESCE(ROUND(ts.total_starts::NUMERIC /
          	NULLIF(ts.days_sampled,0),2),0) AS avg_trips_started,
        	COALESCE(ROUND(te.total_ends::NUMERIC /  
                NULLIF(te.days_sampled,0),2),0) AS avg_trips_ended,
        	COALESCE(ROUND(te.total_ends::NUMERIC /  
                NULLIF(te.days_sampled,0),2),0)  - COALESCE(ROUND(ts.total_starts::NUMERIC /  
                NULLIF(ts.days_sampled,0),2),0) AS avg_net_flow,
        am.avg_bikes_available
    	
    FROM all_tables allt
    LEFT JOIN trip_starts ts ON ts.station_id = allt.station_id AND ts.dow=allt.dow AND ts.hr = allt.hr
    LEFT JOIN trip_ends te ON te.station_id = allt.station_id AND te.dow=allt.dow AND te.hr = allt.hr
    LEFT JOIN availability_metrics am ON am.station_id = allt.station_id AND am.dow=allt.dow AND am.hr = allt.hr
    LIMIT 10;
    ''')
cursor.fetchall()


In [ ]:
conn.close()
print('Connection is closed')